In [1]:
import sys

sys.path.append("..")
import json

import polars as pl

from src.preprocess import extract_relation, reverse_geocode_df, run_cluster, to_csv
from src.preprocess.utils import plot_wordcloud, plot_pov_poi

/home/affahrizain/projects/multi-pov-ir/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
entity_df = (
    pl.read_csv("../dataset/csv/entities.csv")
    .unique(subset=["wikimedia_url"])
    .with_columns(
        pl.col("poi_name_tags")
        .str.split(",")
        .list.eval(pl.element().filter(pl.element() != ""))
    )
    .with_columns(
        pl.col("nearby_pov_cluster")
        .str.split(",")
        .list.eval(pl.element().filter(pl.element() != ""))
        .list.eval(pl.element().cast(pl.Int64))
    )
)

In [3]:
entity_df.filter(
    (pl.col("nearby_pov_cluster").list.len() <= 0)
    | (pl.col("nearby_pov_cluster").is_null())
).filter(pl.col("category") != "other").select(
    ["entity_id", "wikimedia_url", "poi_name", "category", "poi_name_tags"]
).with_columns(pl.col("poi_name_tags").list.join(",")).sort("entity_id").write_csv(
    "../dataset/csv/entities_need_more_scrap.csv"
)

### Bridge

In [4]:
bridge_entity = entity_df.filter(pl.col("category") == "bridge")

Step 1:
- Convert splatone JSON to CSV
- Reverse geocode to obtain detailed place info

In [11]:
data = pl.concat([
    pl.read_csv("../dataset/csv/additional_spots/bridge.csv"),
    pl.read_csv("../dataset/csv/additional_spots/bridge_main.csv").drop("is_recovered")
])
len(data)

26193

In [10]:
# data = json.loads(open("../dataset/splatone/jp-bridge.json", "r").read())
# bridge_df = reverse_geocode_df(to_csv(data))
bridge_df = reverse_geocode_df(data)
bridge_df = bridge_df.filter(pl.col("country_code") == "JP")

Reverse geocoding batches:   0%|          | 0/1 [00:00<?, ?batch/s]

Reverse geocoding batches: 100%|██████████| 1/1 [00:01<00:00,  1.45s/batch]


Step 2:
- Clustering and merge similar ones based on spatial and lexical

In [16]:
ner_labels = ["bridge", "river"]
bridge_spot, bridge_cluster = run_cluster(
    bridge_df, ner_labels, similarity_threshold=0.35
)
bridge_cluster = bridge_cluster.with_columns(
    pl.col("entities").list.eval(pl.element().filter(pl.element() != ""))
)

/home/affahrizain/projects/multi-pov-ir/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
NER: 100%|██████████| 8007/8007 [05:35<00:00, 23.88it/s]


In [19]:
bridge_spot.with_columns(pl.col("entities").list.join(",")).write_csv(
    "../dataset/csv/spots/bridge_spots.csv"
)

Step 3:
- Extract relation between POI (entities) and POV (spots) (also based on spatial and lexical)

In [20]:
bridge_entity = extract_relation(
    bridge_entity, bridge_cluster, similarity_threshold=0.35
)

Step 4:
- Update entity csv (act as db lookup during retrieval)

In [24]:
entity_df = entity_df.update(
    bridge_entity.select(["wikimedia_url", "nearby_pov_cluster"]), on="wikimedia_url"
)
entity_df.with_columns(
    [
        pl.col("poi_name_tags").list.join(","),
        pl.col("nearby_pov_cluster").cast(pl.List(pl.String)).list.join(","),
    ]
).sort("entity_id").write_csv("../dataset/csv/entities.csv")

### Palace/Castle

In [25]:
castle_entity = entity_df.filter(pl.col("category") == "palace_castle")

Step 1:
- Convert splatone JSON to CSV
- Reverse geocode to obtain detailed place info

In [26]:
data = pl.concat([
    pl.read_csv("../dataset/csv/additional_spots/palace_castle.csv"),
    pl.read_csv("../dataset/csv/additional_spots/palace_castle_main.csv").drop("is_recovered")
])
len(data)

40972

In [28]:
# data = json.loads(open("../dataset/splatone/jp-palace-castle.json", "r").read())
# castle_df = reverse_geocode_df(to_csv(data))
castle_df = reverse_geocode_df(data)
castle_df = castle_df.filter(pl.col("country_code") == "JP")

Reverse geocoding batches: 100%|██████████| 1/1 [00:00<00:00,  1.37batch/s]


Step 2:
- Clustering and merge similar ones based on spatial and lexical

In [33]:
ner_labels = ["castle", "城"]
castle_spot, castle_cluster = run_cluster(
    castle_df, ner_labels, similarity_threshold=0.35
)
castle_cluster = castle_cluster.with_columns(
    pl.col("entities").list.eval(pl.element().filter(pl.element() != ""))
)

/home/affahrizain/projects/multi-pov-ir/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
NER: 100%|██████████| 26298/26298 [18:22<00:00, 23.86it/s]


In [37]:
castle_spot.with_columns(pl.col("entities").list.join(",")).write_csv(
    "../dataset/csv/spots/castle_spots.csv"
)

Step 3:
- Extract relation between POI (entities) and POV (spots) (also based on spatial and lexical)

In [38]:
castle_entity = extract_relation(
    castle_entity, castle_cluster, similarity_threshold=0.35
)

Step 4:
- Update entity csv (act as db lookup during retrieval)

In [39]:
entity_df = entity_df.update(
    castle_entity.select(["wikimedia_url", "nearby_pov_cluster"]), on="wikimedia_url"
)
entity_df.with_columns(
    [
        pl.col("poi_name_tags").list.join(","),
        pl.col("nearby_pov_cluster").cast(pl.List(pl.String)).list.join(","),
    ]
).sort("entity_id").write_csv("../dataset/csv/entities.csv")

### Monument/Statue

In [4]:
monument_entity = entity_df.filter(pl.col("category") == "monument_statue")

Step 1:
- Convert splatone JSON to CSV
- Reverse geocode to obtain detailed place info

In [5]:
data = pl.concat([
    pl.read_csv("../dataset/csv/additional_spots/monument_statue.csv"),
    pl.read_csv("../dataset/csv/additional_spots/monument_statue_main.csv")
])
len(data)

12872

In [6]:
# data = json.loads(open("../dataset/splatone/jp-monument-statue.json", "r").read())
# monument_df = reverse_geocode_df(to_csv(data))
monument_df = reverse_geocode_df(data)
monument_df = monument_df.filter(pl.col("country_code") == "JP")

Reverse geocoding batches:   0%|          | 0/1 [00:00<?, ?batch/s]

Reverse geocoding batches: 100%|██████████| 1/1 [00:01<00:00,  1.30s/batch]


Step 2:
- Clustering and merge similar ones based on spatial and lexical

In [13]:
ner_labels = ["monument", "statue", "sculpture", "memorial"]
monument_spot, monument_cluster = run_cluster(
    monument_df, ner_labels, similarity_threshold=0.35
)
monument_cluster = monument_cluster.with_columns(
    pl.col("entities").list.eval(pl.element().filter(pl.element() != ""))
)

/home/affahrizain/projects/multi-pov-ir/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
NER: 100%|██████████| 5428/5428 [03:58<00:00, 22.75it/s]


In [14]:
monument_spot.with_columns(pl.col("entities").list.join(",")).write_csv(
    "../dataset/csv/spots/monument_spots.csv"
)

Step 3:
- Extract relation between POI (entities) and POV (spots) (also based on spatial and lexical)

In [15]:
monument_entity = extract_relation(
    monument_entity, monument_cluster, similarity_threshold=0.35
)

Step 4:
- Update entity csv (act as db lookup during retrieval)

In [16]:
monument_entity.filter(pl.col("nearby_pov_cluster").list.len() <= 0)

entity_id,wikimedia_url,poi_name,wikidata_id,instance_tag,category,poi_name_tags,geohack_url,latitude,longitude,country_code,country,region,subregion,city,osm_id,nearby_pov_cluster
i64,str,str,str,str,str,list[str],str,f64,f64,str,str,str,str,str,str,list[i64]
3976,"""https://commons.wikimedia.org/…","""Naoshima Bath""","""Q11581742""","""[""sentō"", ""work of art""]""","""monument_statue""","[""直島銭湯「i♥湯」"", ""i♥湯"", … ""naoshima bathhouse ""i ♥ yu""""]","""https://geohack.toolforge.org/…",34.458028,133.975194,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Tamano""",null,[]
5301,"""https://commons.wikimedia.org/…","""Peace Memorial Pagoda, Tokushi…","""Q11328768""","""[""pagoda"", ""war memorial""]""","""monument_statue""","[""ハコタ平和記念塔"", ""平和記念塔ハコタ"", … ""peace memorial pagoda""]","""https://geohack.toolforge.org/…",34.066833,134.53775,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Tokushima-shi""",null,[]
5280,"""https://commons.wikimedia.org/…","""Manji-no-sekibutsu""","""Q11353878""","""[""stone Buddha statue""]""","""monument_statue""","[""万治の石仏"", ""manji no sekibutsu"", … ""stone buddha of manji""]","""https://geohack.toolforge.org/…",36.082936,138.081939,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Okaya""","""node/1389203061""",[]
3002,"""https://commons.wikimedia.org/…","""Akashi Municipal Planetarium""","""Q11512514""","""[""astronomical museum"", ""plane…","""monument_statue""","[""明石市立天文科学館"", ""明石天文台"", … ""明石市立天文科学馆""]","""https://geohack.toolforge.org/…",34.649394,135.001478,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Akashi""","""node/1423068720""",[]
5338,"""https://commons.wikimedia.org/…","""Hiroshima Cenotaph""","""Q11409718""","""[""cenotaph""]""","""monument_statue""","[""原爆死没者慰霊碑"", ""広島平和都市記念碑"", … ""cenotaaf voor de atoombomslachtoffers""]","""https://geohack.toolforge.org/…",34.392972,132.452556,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Hiroshima-shi""","""node/1805765497""",[]
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1993,"""https://commons.wikimedia.org/…","""Usuki Stone Buddhas""","""Q2921458""","""[""magaibutsu"", ""sculpture seri…","""monument_statue""","[""bouddhas de pierre de usuki"", ""usuki stone buddhas"", … ""臼杵石佛""]","""https://geohack.toolforge.org/…",33.09011,131.76248,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Usuki""","""node/5804166970""",[]
5695,"""https://commons.wikimedia.org/…","""Saihō-ji (Kyoto)""","""Q46123614""","""[""Buddhist temple"", ""Japanese …","""monument_statue""","[""saihouji"", ""saiho-ji""]","""https://geohack.toolforge.org/…",34.9925,135.684167,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Muko""",null,[]
1678,"""https://commons.wikimedia.org/…","""Chidorigafuchi National Cemete…","""Q2972676""","""[""war cemetery"", ""tomb of the …","""monument_statue""","[""cimetiere national de chidorigafuchi"", ""cimetiere national de la douve de chidori"", … ""nationalfriedhof chidorigafuchi""]","""https://geohack.toolforge.org/…",35.69,139.746944,"""JP""","""Japan""","""Asia""","""Eastern Asia""","""Tokyo""","""way/870979561""",[]


In [17]:
entity_df = entity_df.update(
    monument_entity.select(["wikimedia_url", "nearby_pov_cluster"]), on="wikimedia_url"
)
entity_df.with_columns(
    [
        pl.col("poi_name_tags").list.join(","),
        pl.col("nearby_pov_cluster").cast(pl.List(pl.String)).list.join(","),
    ]
).sort("entity_id").write_csv("../dataset/csv/entities.csv")

### Arch/Gate

In [18]:
gate_entity = entity_df.filter(pl.col("category") == "arch_gate")

Step 1:
- Convert splatone JSON to CSV
- Reverse geocode to obtain detailed place info

In [19]:
data = pl.concat([
    pl.read_csv("../dataset/csv/additional_spots/arch_gate.csv"),
    pl.read_csv("../dataset/csv/additional_spots/arch_gate_main.csv")
])
len(data)

16801

In [20]:
# data = json.loads(open("../dataset/splatone/jp-arch-gate.json", "r").read())
# gate_df = reverse_geocode_df(to_csv(data))
gate_df = reverse_geocode_df(data)
gate_df = gate_df.filter(pl.col("country_code") == "JP")

Reverse geocoding batches: 100%|██████████| 1/1 [00:01<00:00,  1.85s/batch]


Step 2:
- Clustering and merge similar ones based on spatial and lexical

In [21]:
ner_labels = ["gate", "mon", "門", "ゲート", "入り口"]
gate_spot, gate_cluster = run_cluster(gate_df, ner_labels, similarity_threshold=0.35)
gate_cluster = gate_cluster.with_columns(
    pl.col("entities").list.eval(pl.element().filter(pl.element() != ""))
)

/home/affahrizain/projects/multi-pov-ir/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
NER: 100%|██████████| 8218/8218 [06:09<00:00, 22.24it/s]


In [22]:
gate_spot.with_columns(pl.col("entities").list.join(",")).write_csv(
    "../dataset/csv/spots/gate_spots.csv"
)

Step 3:
- Extract relation between POI (entities) and POV (spots) (also based on spatial and lexical)

In [23]:
gate_entity = extract_relation(gate_entity, gate_cluster, similarity_threshold=0.35)

Step 4:
- Update entity csv (act as db lookup during retrieval)

In [24]:
entity_df = entity_df.update(
    gate_entity.select(["wikimedia_url", "nearby_pov_cluster"]), on="wikimedia_url"
)
entity_df.with_columns(
    [
        pl.col("poi_name_tags").list.join(","),
        pl.col("nearby_pov_cluster").cast(pl.List(pl.String)).list.join(","),
    ]
).sort("entity_id").write_csv("../dataset/csv/entities.csv")